In [3]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from dataset import get_flattened_data, class_names
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

BASE_DIR = Path.cwd().parent
RESULTS_DIR = BASE_DIR / "results"



In [5]:
X_train, y_train, X_val, y_val, X_test, y_test = get_flattened_data()

X_combined = np.vstack((X_train, X_val))
y_combined = np.concatenate((y_train, y_val))

test_fold = np.concatenate([
    np.full(len(X_train), -1),
    np.zeros(len(X_val))
])

ps = PredefinedSplit(test_fold)


clf = LogisticRegression(
    solver='saga',
    max_iter=500,
    random_state=42
)

# Use l1_ratio instead of penalty strings.
# 1.0 = L1 (Lasso), 0.0 = L2 (Ridge)
param_grid = {
    'l1_ratio': [1.0, 0.0],
    'C': [0.001, 0.01, 0.1, 1.0]
}

# 3. Use GridSearchCV
grid_search = GridSearchCV(
    clf,
    param_grid,
    cv=ps,
    n_jobs=-2, # Leaves 1 core free
    verbose=1
)

# 4. Fit the model
print("Starting grid search...")
grid_search.fit(X_combined, y_combined)

# 5. Get the best results
best_model = grid_search.best_estimator_
print(f"\nBest Model Parameters: {grid_search.best_params_}")
print(f"Best Cross-Validation Accuracy: {grid_search.best_score_:.4f}")

# Evaluate on your specific validation set
val_predictions = best_model.predict(X_val)
val_accuracy = accuracy_score(y_val, val_predictions)
print(f"Standalone Validation Accuracy: {val_accuracy:.4f}")

Starting grid search...
Fitting 1 folds for each of 8 candidates, totalling 8 fits

Best Model Parameters: {'C': 1.0, 'l1_ratio': 1.0}
Best Cross-Validation Accuracy: 0.8628
Standalone Validation Accuracy: 0.8854


C:\Users\William\PycharmProjects\cs178\.venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [8]:
import pandas as pd
results_df = pd.DataFrame(grid_search.cv_results_)

# Keep only the columns we care about
columns_to_keep = ['param_l1_ratio', 'param_C', 'mean_test_score', 'mean_fit_time']
clean_results = results_df[columns_to_keep].copy()

# Rename the columns so they are easier to read
clean_results.rename(columns={
    'param_l1_ratio': 'Penalty (1=L1, 0=L2)',
    'param_C': 'C',
    'mean_test_score': 'Validation Accuracy',
    'mean_fit_time': 'Train Time (s)'
}, inplace=True)

# Sort the table by the best accuracy at the top
sorted_results = clean_results.sort_values(by='Validation Accuracy', ascending=False)

print("\n--- Detailed Grid Search Results ---")
print(sorted_results.to_string(index=False))


--- Detailed Grid Search Results ---
 Penalty (1=L1, 0=L2)     C  Validation Accuracy  Train Time (s)
                  1.0 1.000             0.862778     1210.325890
                  0.0 0.100             0.862222      168.568051
                  0.0 1.000             0.861222      618.032992
                  1.0 0.100             0.858667      945.144952
                  0.0 0.010             0.857111       74.737880
                  0.0 0.001             0.828111       29.801973
                  1.0 0.010             0.814556      580.967802
                  1.0 0.001             0.660667      247.337197


In [9]:
# Extract results, updating the keys since there is no pipeline anymore
results = [
    {'l1_ratio': params['l1_ratio'], 'C': params['C'], 'accuracy': score}
    for params, score in zip(grid_search.cv_results_['params'], grid_search.cv_results_['mean_test_score'])
]

c_values = [0.001, 0.01, 0.1, 1.0]

# Map the accuracies back to their respective lists using the new l1_ratio values
l1_accuracies = [res['accuracy'] for res in results if res['l1_ratio'] == 1.0]
l2_accuracies = [res['accuracy'] for res in results if res['l1_ratio'] == 0.0]

# --- 2. Validation accuracy vs C value plot ---
plt.plot(c_values, l1_accuracies, marker='o', label='L1 (l1_ratio=1.0)')
plt.plot(c_values, l2_accuracies, marker='s', label='L2 (l1_ratio=0.0)')

plt.xscale('log')
plt.xlabel("Regularization parameter (C)")
plt.ylabel("Validation accuracy")
plt.title("Logistic Regression Validation Accuracy by C and Regularization Type")
plt.legend()
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "lr_c_vs_accuracy.png")
plt.close()

# --- 3. Test evaluation using the best model ---
# Extract best parameters from the new keys
best_l1_ratio = grid_search.best_params_['l1_ratio']
best_c = grid_search.best_params_['C']
best_accuracy = grid_search.best_score_

test_predictions = best_model.predict(X_test)
test_accuracy = accuracy_score(y_test, test_predictions)

# Helper to print/plot the right label
best_label = "L1" if best_l1_ratio == 1.0 else "L2"

# Confusion Matrix
cm = confusion_matrix(y_test, test_predictions)
display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names,
)

fig, ax = plt.subplots(figsize=(10, 8))
display.plot(ax=ax, xticks_rotation=45)
plt.title(f"Logistic Regression Confusion Matrix ({best_label}, C={best_c})")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "lr_confusion_matrix.png")
plt.close()

print()
print(f"Best Regularization: {best_label} (l1_ratio={best_l1_ratio})")
print(f"Best C parameter   : {best_c}")
print(f"Best val accuracy  : {best_accuracy:.4f}")
print(f"Test accuracy      : {test_accuracy:.4f}")


Best Regularization: L1 (l1_ratio=1.0)
Best C parameter   : 1.0
Best val accuracy  : 0.8628
Test accuracy      : 0.8437


In [10]:
from sklearn.metrics import classification_report

# Generate the full classification report
report = classification_report(
    y_test,
    test_predictions,
    target_names=class_names,
    digits=4
)

print("Classification Report:")
print(report)

Classification Report:
              precision    recall  f1-score   support

 T-shirt/top     0.7939    0.8050    0.7994      1000
     Trouser     0.9736    0.9590    0.9662      1000
    Pullover     0.7263    0.7350    0.7306      1000
       Dress     0.8329    0.8670    0.8496      1000
        Coat     0.7350    0.7600    0.7473      1000
      Sandal     0.9429    0.9250    0.9339      1000
       Shirt     0.6310    0.5660    0.5967      1000
     Sneaker     0.9090    0.9390    0.9238      1000
         Bag     0.9284    0.9330    0.9307      1000
  Ankle boot     0.9499    0.9480    0.9489      1000

    accuracy                         0.8437     10000
   macro avg     0.8423    0.8437    0.8427     10000
weighted avg     0.8423    0.8437    0.8427     10000

